# Social Media Engagement Predictor
## End-to-End ML Pipeline · BrightHut / Lighthouse Sanctuary

---

## 1. Problem Framing

### Business Problem

Social media is BrightHut's primary channel for reaching potential donors, but the founders freely admit they are not experienced with it. They post sporadically and struggle with fundamental strategic questions: What content types actually resonate? Which platforms are worth the effort? What time of day should they post? Does social media engagement actually translate into donations — or does it only generate noise?

Without a dedicated marketing team, BrightHut needs the **technology to make smarter decisions** about its social media presence. This pipeline directly addresses that gap by building a system that learns from past post performance to score, rank, and recommend future content decisions.

**The business questions are:**
1. *Can we predict which post attributes (platform, content type, timing, length) are most likely to generate high engagement?* (Predictive)
2. *Which factors are causally associated with higher engagement, and are high-engagement posts also associated with subsequent donation activity?* (Explanatory)

### Who Cares About This

| Stakeholder | How they use this model |
|---|---|
| Executive Director | Understand which platforms and content types are worth investing in |
| Outreach / Comms Staff | Get a pre-post checklist: timing, format, length, and channel recommendations |
| Fundraising Staff | Identify whether high-engagement posts precede donation spikes — close the loop |
| Data Analysts | Monitor model drift over time as the follower base grows |

### Prediction vs. Explanation — Explicit Choice

This pipeline delivers **both** a predictive model and an explanatory model, because the organization has two distinct and non-interchangeable needs:

- **Predictive goal:** Before a post is published, score it on its predicted engagement level. This score is surfaced as a "content recommendation" checklist on the admin dashboard. Out-of-sample classification accuracy (ROC-AUC) is the primary success metric.

- **Explanatory goal:** Quantify which post attributes are most associated with engagement using a regularized OLS regression on log-engagement. Interpretable coefficients and statistical defensibility matter more than marginal accuracy. We explicitly avoid using the predictive ensemble for causal inference.

As the textbook warns: **confusing these goals is a common source of costly analytical mistakes.** We maintain two separate model objects — an OLS regression for explanation, and a gradient boosted classifier for prediction — and interpret each only within its appropriate scope.

### Target Variables

**Primary target — Binary classification (predictive model):**
`high_engagement = 1` if a post's total engagement score (weighted sum of likes, comments, shares) is at or above the 60th percentile of all posts on the same platform. The platform-relative threshold accounts for the fact that Instagram natively drives more reactions than LinkedIn, making cross-platform raw counts misleading.

**Secondary target — Continuous (explanatory model):**
`log_engagement` — natural log of (total_engagement + 1). Log-transformation stabilizes variance and makes the OLS assumption of normality in residuals more defensible.

**Donation-correlation target (supplemental analysis):**
`donation_within_3days = 1` if at least one new donation was recorded within 3 calendar days following the post date. This is a noisy proxy for donation generation — causality cannot be established — but the correlation analysis is strategically useful.

### Success Metrics

| Metric | Rationale |
|---|---|
| ROC-AUC (primary) | Measures discriminative power across all thresholds regardless of class balance |
| Precision @ operating threshold | Recommending a format the org doesn't follow wastes scarce effort |
| Recall @ operating threshold | Missing a high-engagement pattern means missed reach and donor eyeballs |
| OLS adjusted R² | Measures how much variance in log-engagement the explanatory model explains |
| Coefficient p-values | Statistical defensibility of causal-directional claims |

**Error cost asymmetry:** False positives (posting sub-optimal content that was predicted as high-engagement) cost the org time and some credibility. False negatives (avoiding a format that would have performed well) cost reach. Given that posting is low-cost for a small team, **we tune the threshold toward recall** — it is better to try more content than to miss formats that work.

### Important Limitations

Social media performance is influenced by factors the model cannot observe: follower count growth, trending topics, algorithm changes, and external events. The dataset is likely modest in size; results should be treated as directional hypotheses to test rather than definitive rules. **Correlation between posts and donations is not causation** — donors may have already been planning to give before seeing the post. Retraining monthly as new post data accumulates is strongly recommended.

---
## 2. Data Acquisition, Preparation & Exploration

In [ ]:
# ── Install / import ─────────────────────────────────────────────────────────
# Uncomment the line below if running in a fresh Colab / minimal environment
!pip install scikit-learn pandas numpy matplotlib seaborn requests joblib statsmodels --break-system-packages

import warnings, json, re
warnings.filterwarnings('ignore')

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import joblib
from pathlib import Path

from sklearn.pipeline          import Pipeline
from sklearn.preprocessing     import StandardScaler, label_binarize
from sklearn.linear_model      import LogisticRegression, Ridge
from sklearn.ensemble          import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection   import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics           import (
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, fbeta_score, precision_recall_curve,
    mean_squared_error, r2_score
)
from sklearn.inspection        import permutation_importance

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams['figure.dpi'] = 110
sns.set_theme(style='whitegrid', palette='muted')
print('Imports OK')

In [ ]:
# ── Data Loading from local SQLite database ───────────────────────────────────
# Reads directly from the local brighthut.sqlite file — no network required.
# The database file lives at database/brighthut.sqlite relative to the repo root.
# Adjust DB_PATH below if you are running the notebook from a different location.

import sqlite3
import os

DB_PATH = os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'database', 'brighthut.sqlite')
DB_PATH = os.path.normpath(DB_PATH)

if not os.path.exists(DB_PATH):
    raise FileNotFoundError(
        f"Database not found at {DB_PATH}\n"
        "Set DB_PATH to the absolute path of brighthut.sqlite on your machine."
    )

def load_table(name: str) -> pd.DataFrame:
    """Load an entire table from the local SQLite database."""
    with sqlite3.connect(DB_PATH) as conn:
        df = pd.read_sql_query(f"SELECT * FROM {name}", conn)
    print(f'  Loaded {name:<40s} → {df.shape[0]:>4d} rows × {df.shape[1]:>2d} cols')
    return df

print(f'Database: {DB_PATH}')

print('Loading tables...')
posts_raw    = load_table('social_media_posts')
snapshots    = load_table('public_impact_snapshots')
donations    = load_table('donations')
supporters   = load_table('supporters')
print('Done.')


In [ ]:
# ── Schema inspection ─────────────────────────────────────────────────────────
print('=== social_media_posts columns ===')
print(posts_raw.dtypes)
print('\n=== public_impact_snapshots columns ===')
print(snapshots.dtypes)
print('\n=== donations columns (selected) ===')
print(donations.dtypes)
print('\n=== Sample posts row ===')
posts_raw.head(3)

In [ ]:
# ── Helper: resolve dynamic column names ─────────────────────────────────────
def find_col(df, *candidates):
    """Return the first candidate that exists in df columns (case-insensitive). Returns None if none match."""
    cols_lower = {c.lower(): c for c in df.columns}
    for name in candidates:
        if name in df.columns:
            return name
        if name.lower() in cols_lower:
            return cols_lower[name.lower()]
    return None


In [ ]:
# ── Resolve dynamic column names ─────────────────────────────────────────────
# Column names may vary slightly between environments; find_col() handles this.

# social_media_posts
col_post_id      = find_col(posts_raw, 'post_id', 'id')
col_platform     = find_col(posts_raw, 'platform')
col_content_type = find_col(posts_raw, 'content_type', 'type')
col_post_date    = find_col(posts_raw, 'post_date', 'date', 'posted')
col_caption      = find_col(posts_raw, 'caption', 'text', 'content', 'message', 'body')
col_likes        = find_col(posts_raw, 'like', 'likes')
col_comments     = find_col(posts_raw, 'comment', 'comments')
col_shares       = find_col(posts_raw, 'share', 'shares', 'retweet', 'repost')
col_media        = find_col(posts_raw, 'media_type', 'media', 'format')
col_link         = find_col(posts_raw, 'link', 'url')
col_campaign     = find_col(posts_raw, 'campaign', 'campaign_id', 'is_campaign')

# donations
col_don_date     = find_col(donations, 'donation_date', 'date', 'created')
col_don_amount   = find_col(donations, 'amount')

print('Resolved columns:')
mapping = {
    'post_id': col_post_id, 'platform': col_platform,
    'content_type': col_content_type, 'post_date': col_post_date,
    'caption': col_caption, 'likes': col_likes,
    'comments': col_comments, 'shares': col_shares,
    'media': col_media, 'link': col_link, 'campaign': col_campaign,
    'donation_date': col_don_date,
}
for alias, actual in mapping.items():
    print(f'  {alias:20s} → {actual}')

In [ ]:
# ── Build engagement score and targets ───────────────────────────────────────
posts = posts_raw.copy()

# ── 1. Numeric engagement columns (fill missing with 0) ──────────────────────
for col in [col_likes, col_comments, col_shares]:
    if col:
        posts[col] = pd.to_numeric(posts[col], errors='coerce').fillna(0)

# Weighted engagement: shares are highest-value (signals intent to amplify),
# comments indicate conversation, likes indicate passive approval
likes_val    = posts[col_likes].values    if col_likes    else np.zeros(len(posts))
comments_val = posts[col_comments].values if col_comments else np.zeros(len(posts))
shares_val   = posts[col_shares].values   if col_shares   else np.zeros(len(posts))

posts['total_engagement']  = likes_val + (2 * comments_val) + (3 * shares_val)
posts['log_engagement']    = np.log1p(posts['total_engagement'])

print('Engagement distribution:')
print(posts['total_engagement'].describe())

# ── 2. Parse post date ────────────────────────────────────────────────────────
if col_post_date:
    posts['post_date_parsed'] = pd.to_datetime(posts[col_post_date], errors='coerce')
else:
    posts['post_date_parsed'] = pd.NaT

posts = posts.dropna(subset=['post_date_parsed'])
posts = posts.sort_values('post_date_parsed').reset_index(drop=True)

print(f'\nPosts after dropping missing dates: {len(posts)}')

In [ ]:
# ── Platform-relative high-engagement threshold ───────────────────────────────
# Different platforms have different raw engagement volumes; a post that gets
# 50 likes on LinkedIn may be excellent while 50 likes on Instagram may be average.
# We define high_engagement = 1 if the post is >= 60th percentile on its platform.

if col_platform and col_platform in posts.columns:
    posts['platform_clean'] = posts[col_platform].astype(str).str.strip().str.lower()
    thresh = posts.groupby('platform_clean')['total_engagement'].quantile(0.60)
    posts = posts.merge(
        thresh.rename('eng_threshold'), left_on='platform_clean', right_index=True, how='left'
    )
    posts['high_engagement'] = (posts['total_engagement'] >= posts['eng_threshold']).astype(int)
else:
    # Fall back to global 60th percentile if platform is unknown
    global_thresh = posts['total_engagement'].quantile(0.60)
    posts['high_engagement'] = (posts['total_engagement'] >= global_thresh).astype(int)
    posts['platform_clean'] = 'unknown'

print('High-engagement label distribution:')
print(posts['high_engagement'].value_counts(normalize=True).round(3))

In [ ]:
# ── Donation-correlation signal ───────────────────────────────────────────────
# Build a noisy proxy: did any new donation arrive within 3 days of this post?
# IMPORTANT: This is a correlation signal, NOT evidence of causality.
# Donors may have already decided to give before seeing the post.

donation_within_3 = []

if col_don_date and col_don_date in donations.columns:
    donations['don_date_parsed'] = pd.to_datetime(donations[col_don_date], errors='coerce')
    donation_dates = donations['don_date_parsed'].dropna().sort_values().values

    for _, row in posts.iterrows():
        post_dt = row['post_date_parsed']
        window_end = post_dt + pd.Timedelta(days=3)
        in_window = np.sum((donation_dates >= post_dt.to_numpy()) &
                           (donation_dates <= window_end.to_numpy()))
        donation_within_3.append(int(in_window > 0))
else:
    donation_within_3 = [0] * len(posts)

posts['donation_within_3days'] = donation_within_3

print('Posts followed by a donation within 3 days:')
print(posts['donation_within_3days'].value_counts(normalize=True).round(3))

In [ ]:
# ── Feature engineering ───────────────────────────────────────────────────────

# ── Temporal features ────────────────────────────────────────────────────────
posts['day_of_week']  = posts['post_date_parsed'].dt.dayofweek        # 0=Mon, 6=Sun
posts['hour_of_day']  = posts['post_date_parsed'].dt.hour
posts['is_weekend']   = (posts['day_of_week'] >= 5).astype(int)
posts['month']        = posts['post_date_parsed'].dt.month

# Posting cadence: days since last post on same platform (content fatigue proxy)
posts = posts.sort_values('post_date_parsed').reset_index(drop=True)
posts['days_since_last_post'] = (
    posts.groupby('platform_clean')['post_date_parsed']
         .diff()
         .dt.total_seconds()
         .div(86400)
         .fillna(30)          # assume 30-day gap at start
         .clip(upper=90)      # cap outliers at 90 days
)

# ── Caption / text features ───────────────────────────────────────────────────
if col_caption and col_caption in posts.columns:
    text = posts[col_caption].astype(str).fillna('')
    posts['caption_length']   = text.str.len()
    posts['word_count']       = text.str.split().str.len()
    posts['hashtag_count']    = text.str.count(r'#\w+')
    posts['mention_count']    = text.str.count(r'@\w+')
    posts['has_question']     = text.str.contains(r'\?').astype(int)
    posts['has_cta']          = text.str.lower().str.contains(
        r'donate|support|give|help|join|share|link in bio|click'
    ).astype(int)
    posts['mentions_impact']  = text.str.lower().str.contains(
        r'resident|girl|survivor|education|safehouse|outcome|progress|transform'
    ).astype(int)
    posts['mentions_amount']  = text.str.contains(r'\$|\d+%|\d+ (girls|residents|families)').astype(int)
else:
    for c in ['caption_length','word_count','hashtag_count','mention_count',
              'has_question','has_cta','mentions_impact','mentions_amount']:
        posts[c] = 0

# ── Media type features ───────────────────────────────────────────────────────
if col_media and col_media in posts.columns:
    posts['media_clean'] = posts[col_media].astype(str).str.lower().str.strip()
    posts['has_image']   = posts['media_clean'].str.contains('image|photo|picture|img', na=False).astype(int)
    posts['has_video']   = posts['media_clean'].str.contains('video|reel|clip', na=False).astype(int)
else:
    posts['has_image'] = 0
    posts['has_video'] = 0

# ── Link presence ─────────────────────────────────────────────────────────────
posts['has_link'] = 0
if col_link and col_link in posts.columns:
    posts['has_link'] = posts[col_link].notna().astype(int)
elif col_caption and col_caption in posts.columns:
    posts['has_link'] = posts[col_caption].astype(str).str.contains('http', na=False).astype(int)

# ── Campaign flag ─────────────────────────────────────────────────────────────
posts['is_campaign_post'] = 0
if col_campaign and col_campaign in posts.columns:
    posts['is_campaign_post'] = posts[col_campaign].notna().astype(int)

# ── Content type dummies ──────────────────────────────────────────────────────
if col_content_type and col_content_type in posts.columns:
    ct_dummies = pd.get_dummies(posts[col_content_type].astype(str).str.lower().str.strip(),
                                prefix='ctype', drop_first=False, dtype=int)
    posts = pd.concat([posts, ct_dummies], axis=1)
    content_type_cols = list(ct_dummies.columns)
else:
    content_type_cols = []

# ── Platform dummies ──────────────────────────────────────────────────────────
plat_dummies = pd.get_dummies(posts['platform_clean'], prefix='plat', drop_first=False, dtype=int)
posts = pd.concat([posts, plat_dummies], axis=1)
platform_cols = list(plat_dummies.columns)

print(f'Feature engineering complete.  Posts: {len(posts)}')
print(f'Platform dummies: {platform_cols}')
print(f'Content-type dummies: {content_type_cols}')

In [ ]:
# ── Exploratory Data Analysis ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 1. Engagement distribution (raw vs log)
axes[0, 0].hist(posts['total_engagement'], bins=30, edgecolor='white')
axes[0, 0].set_title('Raw Engagement Distribution')
axes[0, 0].set_xlabel('Total Engagement Score')

axes[0, 1].hist(posts['log_engagement'], bins=30, edgecolor='white', color='steelblue')
axes[0, 1].set_title('Log-Engagement Distribution')
axes[0, 1].set_xlabel('log(Engagement + 1)')

# 2. Mean engagement by platform
if len(posts['platform_clean'].unique()) > 1:
    plat_eng = posts.groupby('platform_clean')['total_engagement'].mean().sort_values(ascending=False)
    plat_eng.plot(kind='bar', ax=axes[0, 2], color='coral', edgecolor='white')
    axes[0, 2].set_title('Avg Engagement by Platform')
    axes[0, 2].set_xticklabels(axes[0, 2].get_xticklabels(), rotation=30, ha='right')

# 3. Mean engagement by day of week
day_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
day_eng = posts.groupby('day_of_week')['total_engagement'].mean()
day_eng.index = [day_names[i] for i in day_eng.index]
day_eng.plot(kind='bar', ax=axes[1, 0], color='mediumseagreen', edgecolor='white')
axes[1, 0].set_title('Avg Engagement by Day of Week')
axes[1, 0].set_xticklabels(axes[1, 0].get_xticklabels(), rotation=0)

# 4. Mean engagement by hour of day
hour_eng = posts.groupby('hour_of_day')['total_engagement'].mean()
hour_eng.plot(kind='line', ax=axes[1, 1], marker='o', color='mediumpurple')
axes[1, 1].set_title('Avg Engagement by Hour of Day')
axes[1, 1].set_xlabel('Hour (24h)')

# 5. Caption length vs engagement
axes[1, 2].scatter(posts['caption_length'].clip(upper=1500),
                   posts['log_engagement'], alpha=0.4, s=20)
axes[1, 2].set_title('Caption Length vs Log-Engagement')
axes[1, 2].set_xlabel('Caption Length (chars)')
axes[1, 2].set_ylabel('log(Engagement)')

plt.suptitle('Social Media Post EDA — BrightHut', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation heatmap of numeric features ───────────────────────────────────
numeric_cols = [
    'log_engagement', 'day_of_week', 'hour_of_day', 'is_weekend',
    'days_since_last_post', 'caption_length', 'word_count',
    'hashtag_count', 'mention_count', 'has_question', 'has_cta',
    'mentions_impact', 'mentions_amount', 'has_image', 'has_video',
    'has_link', 'is_campaign_post'
]
available = [c for c in numeric_cols if c in posts.columns]

corr = posts[available].corr()
plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.4, annot_kws={'size': 7})
plt.title('Feature Correlation Matrix (Social Media Posts)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Donation-correlation analysis ─────────────────────────────────────────────
print('=== Posts followed by donation within 3 days ===')
print(posts['donation_within_3days'].value_counts())

# Are high-engagement posts more likely to be followed by a donation?
contingency = pd.crosstab(posts['high_engagement'], posts['donation_within_3days'],
                          rownames=['high_engagement'], colnames=['donation_within_3days'])
print('\nContingency table (high_engagement vs donation_within_3days):')
print(contingency)

# Conditional rates
print('\nDonation-following rate by high_engagement label:')
print(posts.groupby('high_engagement')['donation_within_3days'].mean().round(3))

---
## 3. Modeling & Feature Selection

In [ ]:
# ── Assemble feature matrix ───────────────────────────────────────────────────

BASE_FEATURES = [
    # Temporal
    'day_of_week', 'hour_of_day', 'is_weekend', 'month', 'days_since_last_post',
    # Text signals
    'caption_length', 'word_count', 'hashtag_count', 'mention_count',
    'has_question', 'has_cta', 'mentions_impact', 'mentions_amount',
    # Media
    'has_image', 'has_video', 'has_link',
    # Campaign
    'is_campaign_post',
]

ALL_FEATURE_COLS = BASE_FEATURES + platform_cols + content_type_cols
ALL_FEATURE_COLS = [c for c in ALL_FEATURE_COLS if c in posts.columns]

# Fill any remaining NaN in features with 0
posts[ALL_FEATURE_COLS] = posts[ALL_FEATURE_COLS].fillna(0)

X = posts[ALL_FEATURE_COLS].copy()
y = posts['high_engagement'].copy()

print(f'Feature matrix shape:  {X.shape}')
print(f'Target class balance:\n{y.value_counts(normalize=True).round(3)}')

In [ ]:
# ── Chronological train/test split ────────────────────────────────────────────
# Social media data is time-ordered; we use the most recent 20% as the test set
# to simulate true out-of-sample performance. Random splits would leak future
# temporal patterns into training, inflating estimates.

split_idx = int(len(posts) * 0.80)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f'Train: {len(X_train)} posts  |  Test: {len(X_test)} posts')
print(f'Train high_eng rate: {y_train.mean():.3f}  |  Test high_eng rate: {y_test.mean():.3f}')

# Cross-validation uses StratifiedKFold on train set only
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [ ]:
# ── Model A: Logistic Regression (EXPLANATORY) ────────────────────────────────
# Interpretable coefficients and odds ratios; used for causal-directional inference.
# L2 regularization (C=1.0) prevents overfitting while retaining coefficient stability.

pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(
                   class_weight='balanced',
                   max_iter=1000,
                   C=1.0,
                   random_state=RANDOM_STATE
               ))
])

cv_scores_lr = cross_val_score(pipe_lr, X_train, y_train,
                               cv=cv, scoring='roc_auc', n_jobs=-1)
print(f'LR  CV ROC-AUC: {cv_scores_lr.mean():.4f}  (±{cv_scores_lr.std():.4f})')
pipe_lr.fit(X_train, y_train)

In [ ]:
# ── Model B: Random Forest Classifier (PREDICTIVE) ───────────────────────────
# Captures non-linear interactions (e.g., video content on Instagram performs
# differently than video content on LinkedIn). No coefficient interpretability.

pipe_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    RandomForestClassifier(
                   n_estimators=400,
                   max_depth=6,
                   min_samples_leaf=3,
                   class_weight='balanced',
                   random_state=RANDOM_STATE,
                   n_jobs=-1
               ))
])

cv_scores_rf = cross_val_score(pipe_rf, X_train, y_train,
                               cv=cv, scoring='roc_auc', n_jobs=-1)
print(f'RF  CV ROC-AUC: {cv_scores_rf.mean():.4f}  (±{cv_scores_rf.std():.4f})')
pipe_rf.fit(X_train, y_train)

In [ ]:
# ── Model C: Gradient Boosted Trees (PREDICTIVE — primary) ───────────────────
# Sequentially corrects residuals; typically achieves best out-of-sample AUC
# on tabular data. Used as the deployed inference model.

pipe_gbt = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    GradientBoostingClassifier(
                   n_estimators=300,
                   learning_rate=0.05,
                   max_depth=4,
                   subsample=0.8,
                   random_state=RANDOM_STATE
               ))
])

cv_scores_gbt = cross_val_score(pipe_gbt, X_train, y_train,
                                cv=cv, scoring='roc_auc', n_jobs=-1)
print(f'GBT CV ROC-AUC: {cv_scores_gbt.mean():.4f}  (±{cv_scores_gbt.std():.4f})')
pipe_gbt.fit(X_train, y_train)

---
## 4. Evaluation & Selection

In [ ]:
# ── Hold-out test performance ─────────────────────────────────────────────────
results = {}
for name, pipe in [('Logistic Regression', pipe_lr),
                   ('Random Forest',       pipe_rf),
                   ('Gradient Boosting',   pipe_gbt)]:
    proba = pipe.predict_proba(X_test)[:, 1]
    auc   = roc_auc_score(y_test, proba)
    results[name] = {'ROC-AUC': round(auc, 4), 'proba': proba}
    print(f'{name:25s}  Test AUC = {auc:.4f}')

best_model_name = max({k: v['ROC-AUC'] for k, v in results.items()}, key=lambda k: results[k]['ROC-AUC'])
best_pipe       = {'Logistic Regression': pipe_lr,
                   'Random Forest':       pipe_rf,
                   'Gradient Boosting':   pipe_gbt}[best_model_name]
best_proba      = results[best_model_name]['proba']

print(f'\nBest predictive model: {best_model_name}')

In [ ]:
# ── Threshold tuning with F-beta (beta=1.5) ───────────────────────────────────
# Missing a genuinely high-engagement opportunity is costly (lost reach, donor eyeballs)
# so we weight recall slightly more than precision. Beta=1.5 balances this.

precisions, recalls, thresholds = precision_recall_curve(y_test, best_proba)
f_beta = [fbeta_score(y_test, (best_proba >= t).astype(int), beta=1.5)
          for t in thresholds]

best_idx      = int(np.argmax(f_beta))
BEST_THRESHOLD = float(thresholds[best_idx])

print(f'Optimal threshold (max F1.5): {BEST_THRESHOLD:.3f}')
print(f'  Precision: {precisions[best_idx]:.3f}')
print(f'  Recall:    {recalls[best_idx]:.3f}')
print(f'  F1.5:      {f_beta[best_idx]:.3f}')

plt.figure(figsize=(8, 4))
plt.plot(thresholds, f_beta, color='teal')
plt.axvline(BEST_THRESHOLD, color='red', linestyle='--',
            label=f'Best threshold = {BEST_THRESHOLD:.3f}')
plt.xlabel('Decision Threshold')
plt.ylabel('F1.5 Score')
plt.title('F1.5-Optimized Threshold — Engagement Classifier')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion matrix at operating threshold ───────────────────────────────────
y_pred_thresh = (best_proba >= BEST_THRESHOLD).astype(int)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_thresh,
    display_labels=['Low Engagement', 'High Engagement'],
    cmap='Blues', ax=ax
)
ax.set_title(f'{best_model_name} @ threshold={BEST_THRESHOLD:.2f}')
plt.tight_layout()
plt.show()

print('\nClassification Report:')
print(classification_report(y_test, y_pred_thresh,
                            target_names=['Low Engagement', 'High Engagement']))

In [ ]:
# ── Permutation importance on held-out test set ───────────────────────────────
# Permutation importance on the test set avoids in-bag bias that affects
# impurity-based importance from tree models.

perm_result = permutation_importance(
    best_pipe, X_test, y_test,
    n_repeats=20, scoring='roc_auc',
    random_state=RANDOM_STATE, n_jobs=-1
)

perm_df = pd.DataFrame({
    'feature':    ALL_FEATURE_COLS,
    'importance': perm_result.importances_mean,
    'std':        perm_result.importances_std
}).sort_values('importance', ascending=False).head(20)

plt.figure(figsize=(9, 6))
plt.barh(perm_df['feature'][::-1], perm_df['importance'][::-1],
         xerr=perm_df['std'][::-1], color='darkorange', alpha=0.85)
plt.xlabel('Mean Decrease in ROC-AUC')
plt.title(f'Top Feature Importances (Permutation) — {best_model_name}')
plt.tight_layout()
plt.show()

print('\nTop 10 features by permutation importance:')
print(perm_df.head(10).to_string(index=False))

---
## 5. Causal and Relationship Analysis

### 5a. Explanatory OLS Regression on Log-Engagement

We use a regularized OLS regression (`statsmodels` OLS with the full feature set) to obtain interpretable coefficient estimates. Unlike the GBT/RF ensemble, OLS produces effect sizes with confidence intervals and p-values, which are the appropriate tools for answering: *"Holding everything else constant, how much does posting a video vs. an image change expected engagement?"*

**Tautological features excluded from the explanatory model:**
- Raw like/comment/share counts — these ARE the target variable, definitionally
- `total_engagement` and `log_engagement` — the outcome itself

All other features are retained because they represent pre-publish decisions the organization can actually control.

In [ ]:
# ── OLS regression on log_engagement (explanatory) ───────────────────────────

# Outcome: log_engagement (continuous, normally distributed after log transform)
y_ols = posts['log_engagement']

# Explanatory features: everything except outcome-linked and post-publish variables
EXCLUDE_FROM_OLS = {'log_engagement', 'total_engagement', 'high_engagement',
                    'donation_within_3days'}
ols_features = [c for c in ALL_FEATURE_COLS if c not in EXCLUDE_FROM_OLS]

X_ols = posts[ols_features].copy().fillna(0)
X_ols_scaled = (X_ols - X_ols.mean()) / (X_ols.std().replace(0, 1))  # standardize for comparability
X_ols_c = sm.add_constant(X_ols_scaled)

ols_model = sm.OLS(y_ols, X_ols_c).fit()
print(ols_model.summary())

In [ ]:
# ── Plot OLS coefficients with 95% CI ────────────────────────────────────────
coef_df = pd.DataFrame({
    'feature': ols_model.params.index,
    'coef':    ols_model.params.values,
    'ci_lo':   ols_model.conf_int()[0].values,
    'ci_hi':   ols_model.conf_int()[1].values,
    'pvalue':  ols_model.pvalues.values
}).query("feature != 'const'")

# Show top predictors by absolute coefficient
coef_df['abs_coef'] = coef_df['coef'].abs()
coef_df = coef_df.sort_values('abs_coef', ascending=False).head(20)

colors = ['steelblue' if c > 0 else 'tomato' for c in coef_df['coef']]

plt.figure(figsize=(10, 7))
plt.barh(coef_df['feature'][::-1], coef_df['coef'][::-1], color=colors[::-1], alpha=0.85)
plt.errorbar(
    coef_df['coef'][::-1],
    range(len(coef_df)),
    xerr=[
        (coef_df['coef'] - coef_df['ci_lo'])[::-1].values,
        (coef_df['ci_hi'] - coef_df['coef'])[::-1].values
    ],
    fmt='none', color='black', capsize=3, linewidth=1
)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Standardized OLS Coefficient (effect on log-engagement)')
plt.title('Explanatory Model: Feature Effects on Post Engagement\n(Standardized — comparable across features)')
plt.tight_layout()
plt.show()

print('\nStatistically significant features (p < 0.10):')
sig = coef_df[coef_df['pvalue'] < 0.10].sort_values('coef', ascending=False)
print(sig[['feature','coef','pvalue']].to_string(index=False))

In [ ]:
# ── 5b. Donation-correlation logistic regression ──────────────────────────────
# Secondary question: are high-engagement posts more likely to be followed by a
# donation? We fit a logistic regression predicting donation_within_3days.
# CAUTION: This is correlational; we cannot claim the post CAUSED the donation.

don_features = ['high_engagement', 'is_campaign_post', 'mentions_impact',
                'mentions_amount', 'has_cta', 'is_weekend'] + platform_cols
don_features = [c for c in don_features if c in posts.columns]

y_don = posts['donation_within_3days']
X_don = posts[don_features].fillna(0)

if y_don.sum() >= 5 and (len(y_don) - y_don.sum()) >= 5:  # guard tiny sample
    X_don_c = sm.add_constant(X_don)
    try:
        logit_don = sm.Logit(y_don, X_don_c).fit(disp=False, maxiter=200)
        print(logit_don.summary())

        # Odds ratios
        or_df = pd.DataFrame({
            'feature':    logit_don.params.index,
            'odds_ratio': np.exp(logit_don.params.values),
            'pvalue':     logit_don.pvalues.values
        }).query("feature != 'const'").sort_values('odds_ratio', ascending=False)
        print('\nOdds ratios for donation-within-3days:')
        print(or_df.to_string(index=False))
    except Exception as e:
        print(f'Donation logit failed (possibly singular matrix with small sample): {e}')
else:
    print('Insufficient donation-followed posts for logistic regression — skipping.')
    print('This is expected with a small dataset; report raw rates instead.')

In [ ]:
# ── 5c. Strategic timing and platform analysis ────────────────────────────────
# Actionable summary: best day + time combinations by platform

print('=== High-engagement rate by platform ===')
plat_summary = posts.groupby('platform_clean').agg(
    n_posts=('high_engagement', 'count'),
    high_eng_rate=('high_engagement', 'mean'),
    avg_engagement=('total_engagement', 'mean')
).sort_values('high_eng_rate', ascending=False)
print(plat_summary.round(3))

print('\n=== Best day of week per platform ===')
for plat in posts['platform_clean'].unique():
    subset = posts[posts['platform_clean'] == plat]
    if len(subset) < 5:
        continue
    best_day = subset.groupby('day_of_week')['total_engagement'].mean().idxmax()
    best_hr  = subset.groupby('hour_of_day')['total_engagement'].mean().idxmax()
    print(f'  {plat:12s}  best day={day_names[best_day]:3s}  best hour={best_hr:02d}:00')

print('\n=== High-engagement rate: video vs image vs text ===')
for col, flag_col in [('has_video', 'Video'), ('has_image', 'Image')]:
    rate_1 = posts[posts[col] == 1]['high_engagement'].mean() if col in posts else float('nan')
    rate_0 = posts[posts[col] == 0]['high_engagement'].mean() if col in posts else float('nan')
    print(f'  {flag_col:6s}: with={rate_1:.3f}  without={rate_0:.3f}')

---
## 6. Deployment Notes

### How It Was Actually Deployed

The social media engagement findings were deployed as **static hardcoded guidance** in the React frontend — no dynamic API endpoint or model-scoring service was created. The OLS coefficient findings and best-practice recommendations from this notebook were translated directly into a collapsible **Outreach Playbook** panel in the Social Media page.

#### Frontend Integration

**`frontend/src/pages/SocialPortal.tsx`** — the Social Media page (route: `/social`).

The Outreach Playbook is a collapsible panel (toggle button at the top) containing three card groups derived from this notebook's findings alongside the campaign-effectiveness insights:

| Card Group | Insights Shown |
|---|---|
| **What to Post** | Content type recommendations (video/image preference, calls-to-action, impact-anchored language) |
| **When to Post** | Platform-optimal timing, posting cadence, frequency guidelines |
| **Campaign Strategy** | Cross-pipeline strategic guidance combining engagement + revenue findings |

**Where to find it in the app:** Navigate to **Social Media** (from the admin sidebar). Click the **"Outreach Playbook ▾"** toggle at the top of the page to expand the guidance panel.

#### Why No Dynamic Endpoint?

Social media post data is at the aggregate/post level (not per-resident or per-donor), and real-time per-post scoring would require a publishing integration that is out of scope. The OLS explanatory coefficients provide reliable directional guidance that does not need to be recomputed per-request. Hardcoding the findings avoids over-engineering a scoring API for static strategic advice.

#### To Update the Guidance

If new post engagement data meaningfully changes the OLS coefficient directions (e.g., video stops outperforming image posts), update the static card content in the `sp-playbook-body` section of `SocialPortal.tsx` to reflect the revised findings.


In [ ]:
# ── Save model artifacts ──────────────────────────────────────────────────────
ARTIFACT_DIR = Path('model_artifacts')
ARTIFACT_DIR.mkdir(exist_ok=True)

model_path  = ARTIFACT_DIR / 'social_engagement_model.pkl'
config_path = ARTIFACT_DIR / 'social_engagement_config.json'

joblib.dump(best_pipe, model_path)
print(f'Model saved → {model_path}')

config = {
    'model_type':        best_model_name,
    'threshold':         round(BEST_THRESHOLD, 4),
    'feature_cols':      ALL_FEATURE_COLS,
    'platform_cols':     platform_cols,
    'content_type_cols': content_type_cols,
    'target':            'high_engagement',
    'notes':             'Platform-relative 60th-percentile threshold. Retrain monthly.',
    'api_endpoint':      'GET /api/ml/post-engagement',
    'auth_required':     'admin'
}

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f'Config saved → {config_path}')

In [ ]:
# ── Inference function for API integration ────────────────────────────────────

def score_post(post_metadata: dict) -> dict:
    """
    Score a candidate social media post before it is published.

    Parameters
    ----------
    post_metadata : dict
        Must contain the same keys used in ALL_FEATURE_COLS.
        Keys not provided default to 0.

    Returns
    -------
    dict with keys:
        - engagement_score  : float  probability of high engagement (0–1)
        - high_engagement   : int    1 = predicted high, 0 = predicted low
        - recommendation    : str    human-readable summary
    """
    with open(config_path) as f:
        cfg = json.load(f)

    model = joblib.load(model_path)
    row   = {col: post_metadata.get(col, 0) for col in cfg['feature_cols']}
    X_new = pd.DataFrame([row])

    score = float(model.predict_proba(X_new)[0, 1])
    label = int(score >= cfg['threshold'])

    if label == 1:
        rec = (f'High engagement predicted (score={score:.2f}). '
               'This post configuration is likely to perform well — proceed as planned.')
    else:
        rec = (f'Low engagement predicted (score={score:.2f}). '
               'Consider adjusting: add a call-to-action, include a video/image, '
               'post on a higher-performing day, or reference a specific impact story.')

    return {
        'engagement_score': round(score, 4),
        'high_engagement':  label,
        'recommendation':   rec
    }


# ── Example usage ─────────────────────────────────────────────────────────────
example = {
    'day_of_week': 5,        # Saturday
    'hour_of_day': 10,       # 10am
    'is_weekend': 1,
    'has_video': 1,
    'has_cta': 1,
    'mentions_impact': 1,
    'caption_length': 280,
    'hashtag_count': 5,
    'is_campaign_post': 1
}

result = score_post(example)
print('\nExample post score:')
for k, v in result.items():
    print(f'  {k}: {v}')